# 3.12 · 流水线 / Pipelines —— Part 3 收官 🏁

> **课程定位 / Where this fits**
> **Part 3 最后一课**。前 11 课每节都在喊"用 Pipeline 防泄漏"——现在兑现承诺。Pipeline 把**填补→缩放→编码→建模**串成一个对象, 让"只 fit train"成为**结构性保证而非手动纪律**。这也是模型**从 notebook 走向生产**的桥梁。
> Pipelines turn "fit only on train" from manual discipline into a structural guarantee, and bridge notebook to production.

> 💡 **面试相关 / Interview-relevant**
> - "为什么用 Pipeline" ★★★★★（防泄漏 + 可部署 + 可调参）
> - "ColumnTransformer 干什么" ★★★★（不同列不同处理）
> - "Pipeline 怎么和 GridSearch 配合" ★★★★
> - "怎么把整个预处理 + 模型一起保存上线" ★★★

---

## 学习目标 / Learning Objectives
1. 理解 Pipeline 如何**结构性消除泄漏**（fit 自动只在 train）。
2. 用 **ColumnTransformer** 对数值/类别列分别处理。
3. 把 Pipeline 接进 **GridSearchCV** 调全流程超参。
4. **一个对象保存→加载→部署**整条流水线。
5. 整合 Part 3 全部技能, 在 Titanic 上端到端建一个**可上线**的模型。

## 目录 / TOC
1. [为什么 Pipeline ⭐](#1)
2. [最小 Pipeline](#2)
3. [ColumnTransformer：分列处理 ⭐](#3)
4. [Pipeline + GridSearchCV 调参 ⭐](#4)
5. [保存与部署](#5)
6. [实战：Titanic 端到端可上线模型 ⭐](#6)
7. [小结 + Part 3 总结 🏁](#7)


<a id="1"></a>
## 1. 为什么 Pipeline ⭐ / Why Pipelines

手动预处理的三宗罪 vs Pipeline 的三个解决：

| 手动的问题 | Pipeline 的解决 |
|---|---|
| **泄漏**：容易手滑在全数据 fit | `fit()` 自动只在传入的 train 上做, CV 每折自动重新 fit |
| **不可部署**：上线要重写一遍预处理 | 整条流水线一个对象, `joblib.dump` 直接存 |
| **难调参**：预处理参数 + 模型参数分散 | GridSearchCV 一次调全流程（含预处理超参）|

**核心机制**：Pipeline = 一串 (名字, 转换器) + 最后一个估计器。
- `pipe.fit(X_tr, y_tr)`：依次 `fit_transform` 每个转换器, 最后 `fit` 模型
- `pipe.predict(X_te)`：依次 `transform`（**不 fit!**）, 最后 `predict`

→ **test 永远只被 transform, 从不参与 fit** = 泄漏在结构上不可能发生。
test is only ever transformed, never fit — leakage becomes structurally impossible.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
rng = np.random.default_rng(42)
print("imports ok")


<a id="2"></a>
## 2. 最小 Pipeline / Minimal Pipeline

把"填补 → 缩放 → 模型"串成一个对象。


In [ ]:
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X = X.copy()
X.iloc[::20, 0] = np.nan        # 人为制造一些缺失 / inject missingness

# 三步流水线 / three-step pipeline
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000)),
])

# CV: 每折自动重新 fit 整条流水线 (impute/scale 只在每折 train) / leak-free by construction
scores = cross_val_score(pipe, X, y, cv=5)
print(f"Pipeline CV 准确率: {scores.mean():.1%} ± {scores.std():.1%}")
print("\n关键: cross_val_score 对每个 fold:")
print("  1. 在 train 折上 fit impute+scale+clf")
print("  2. 在 val 折上只 transform+predict")
print("  → impute 的中位数、scale 的 μ/σ 都只来自 train 折, 零泄漏 (无需手动操心)")


<a id="3"></a>
## 3. ColumnTransformer：分列处理 ⭐ / ColumnTransformer

**真实数据混合数值和类别列**——需要分别处理：数值列填中位数+缩放, 类别列填众数+one-hot。`ColumnTransformer` 让不同列走不同子流水线。


In [ ]:
df = sns.load_dataset("titanic")
# 选特征 (删掉泄漏列 alive, 高缺失 deck) / drop leaky/high-missing columns
features = ["pclass","sex","age","sibsp","parch","fare","embarked"]
X = df[features].copy()
y = df["survived"]

num_cols = ["age","sibsp","parch","fare"]       # 数值
cat_cols = ["pclass","sex","embarked"]           # 类别 (pclass 当类别处理也行)

# 数值子流水线: 中位数填补 + 标准化 / numeric sub-pipeline
num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
# 类别子流水线: 众数填补 + one-hot / categorical sub-pipeline
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])

# 组合: 不同列走不同处理 / combine
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

# 完整流水线: 预处理 + 模型 / full pipeline
full = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))])
print(f"ColumnTransformer + LogReg CV: {cross_val_score(full, X, y, cv=5).mean():.1%}")
print(f"\n处理后特征数: {preprocess.fit_transform(X).shape[1]} "
      f"(4 数值 + one-hot 展开的类别列)")


**这一个 `full` 对象就封装了 3.2(填补) + 3.4(缩放) + 3.5(编码) + 建模的全部**——而且全程结构性防泄漏。这就是 Part 3 所有技能的集大成。
This single `full` object encapsulates imputation + scaling + encoding + modeling, all leak-free by construction.


<a id="4"></a>
## 4. Pipeline + GridSearchCV 调参 ⭐ / Tuning the Whole Pipeline

**威力所在**：GridSearchCV 能同时调**预处理超参 + 模型超参**, 用 `步骤名__参数名` 双下划线语法访问嵌套参数。


In [ ]:
# 调参: 填补策略 + 模型 C, 一起搜 / tune imputation strategy AND model C together
full = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))])

param_grid = {
    "prep__num__impute__strategy": ["median", "mean"],     # 数值填补策略 (嵌套3层!)
    "clf__C": [0.1, 1.0, 10.0],                             # 模型正则强度
}
grid = GridSearchCV(full, param_grid, cv=5, scoring="accuracy")
grid.fit(X, y)

print(f"最优参数: {grid.best_params_}")
print(f"最优 CV 准确率: {grid.best_score_:.1%}")
print("\n双下划线语法: prep__num__impute__strategy")
print("  prep(ColumnTransformer) → num(数值子流水线) → impute(填补器) → strategy(参数)")
print("→ 整个流程的任何超参都能一起调, 且每个组合都在 CV 折内诚实评估")


<a id="5"></a>
## 5. 保存与部署 / Save & Deploy

**一个对象, 一行保存**——预处理 + 模型一起。上线时加载即用, **输入原始数据**, Pipeline 自动跑完整条流程。


In [ ]:
import joblib
from pathlib import Path

# 训练最终模型 (用全部数据) / train final model on all data
final_model = grid.best_estimator_
final_model.fit(X, y)

# 保存整条流水线 / save the whole pipeline
model_path = "/tmp/titanic_pipeline.joblib"
joblib.dump(final_model, model_path)
print(f"已保存: {model_path} ({Path(model_path).stat().st_size/1024:.0f} KB)")

# 模拟部署: 加载 + 对原始新数据预测 / deploy: load + predict on RAW new data
loaded = joblib.load(model_path)
new_passenger = pd.DataFrame([{
    "pclass": 1, "sex": "female", "age": 28, "sibsp": 0,
    "parch": 0, "fare": 80.0, "embarked": "S"
}])
# 注意: 直接喂原始数据! Pipeline 自动填补/缩放/编码 / raw data in, Pipeline handles everything
prob = loaded.predict_proba(new_passenger)[0, 1]
print(f"\n新乘客 (一等舱女性): 生还概率 = {prob:.1%}")
print("→ 部署只需: load + predict(原始数据). 预处理逻辑封装在内, 永不会'忘了缩放'")


> 💡 **为什么这是生产级的**：传统做法上线要把 notebook 里的预处理代码**重抄一遍**到服务里——极易和训练时不一致（训练用中位数 28, 服务里手写成 30）= **training-serving skew**, 隐蔽的线上事故。Pipeline 把训练和服务的预处理**绑成同一份代码**, 根除这类 bug。
> Pipelines bind training and serving preprocessing into one artifact, eliminating training-serving skew.


<a id="6"></a>
## 6. 实战：Titanic 端到端可上线模型 ⭐ / End-to-end Deployable Model

**整合 Part 3 全部技能**：EDA 洞察(3.1) + 特征工程(3.6) + 填补(3.2) + 编码(3.5) + 缩放(3.4) + 防泄漏(3.9) + 正确 CV(3.10) + Pipeline(3.12)。


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# 自定义转换器: 加 3.1/3.6 的工程特征 (family_size, is_alone) / custom feature engineering step
class TitanicFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X["family_size"] = X["sibsp"] + X["parch"] + 1            # 3.6 特征工程
        X["is_alone"] = (X["family_size"] == 1).astype(int)
        X["fare_log"] = np.log1p(X["fare"].fillna(X["fare"].median()))  # 3.6 log 变换
        return X

# 重新取原始特征 / fresh raw features
X = df[["pclass","sex","age","sibsp","parch","fare","embarked"]].copy()
y = df["survived"]

num_cols = ["age","sibsp","parch","fare","family_size","fare_log"]
cat_cols = ["pclass","sex","embarked","is_alone"]

num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

# 完整可上线流水线: 特征工程 → 预处理 → 模型 / fully deployable
deployable = Pipeline([
    ("features", TitanicFeatures()),       # 3.6
    ("prep", prep),                        # 3.2 + 3.4 + 3.5
    ("clf", RandomForestClassifier(n_estimators=200, random_state=0)),
])

# 诚实评估 (分层 CV, 3.10) / honest evaluation with stratified CV
from sklearn.model_selection import StratifiedKFold
scores = cross_val_score(deployable, X, y, cv=StratifiedKFold(5), scoring="accuracy")
print(f"端到端流水线 CV 准确率: {scores.mean():.1%} ± {scores.std():.1%}")

deployable.fit(X, y)
print(f"\n这一个 'deployable' 对象包含了 Part 3 的全部:")
print("  TitanicFeatures (3.1 EDA洞察 + 3.6 特征工程)")
print("  → 数值: 中位数填补(3.2) + 标准化(3.4)")
print("  → 类别: 众数填补(3.2) + one-hot(3.5)")
print("  → 随机森林; 全程分层CV评估(3.10), 结构防泄漏(3.9/3.12)")
print("  joblib.dump 一行即可上线")


<a id="7"></a>
## 7. 小结 + Part 3 总结 🏁 / Summary

### Pipeline 小结
```
Pipeline = 转换器链 + 估计器; fit 只在 train, predict 只 transform → 结构防泄漏 ⭐
ColumnTransformer: 数值/类别列走不同子流水线
GridSearchCV: 双下划线 step__param 调全流程超参 (含预处理)
joblib.dump: 一个对象保存整条流水线 → 根除 training-serving skew
```

---

## 🏁 Part 3 全部完成 / Part 3 Complete!

| # | 课 | 核心带走 |
|---|---|---|
| 3.1 | EDA | 五步清单 + quality_report + 交互效应 |
| 3.2 | 缺失值 | MCAR/MAR/MNAR + 均值填补两宗罪 |
| 3.3 | 异常值 | 三种身份 + z自我掩盖 + IsolationForest |
| 3.4 | 特征缩放 | 距离/梯度要缩放, 树不用 + RobustScaler |
| 3.5 | 类别编码 | 名义vs有序 + target编码K-fold防泄漏 |
| 3.6 | 特征工程 | 比率/分箱/sin-cos周期/聚合 |
| 3.7 | 文本特征 | 词袋 → TF-IDF → n-gram |
| 3.8 | 图像特征 | 像素弱点 + HOG + CNN自动化 |
| 3.9 | 数据泄漏 ⭐ | 五形态 + 噪声虚高震撼演示 |
| 3.10 | 划分与CV | 分层/分组/时序 + nested CV |
| 3.11 | 不平衡 | 准确率悖论 + SMOTE泄漏 + 阈值调整 |
| 3.12 | Pipeline | ColumnTransformer + 结构防泄漏 + 部署 |

**两条贯穿线**：
1. **防泄漏是 Part 3 的灵魂**——3.2/3.4/3.5/3.6/3.9/3.10/3.11 节节都在防, 3.12 用 Pipeline 一劳永逸
2. **"清洗→变换→建模"必须是一个整体**——Pipeline 让它既防泄漏又可部署

### 下一站
**Part 4 · 监督学习：回归**——数据预处理武器库就位, 终于开始**正式建模**！从线性回归的数学推导到 XGBoost。
